# SimCLR Contrastive Training for WISDM Dataset


In [ ]:
# Author: C. I. Tang
# Adapted for WISDM by Mochi
# Based on work of Tang et al.: https://arxiv.org/abs/2011.11542
# License: GNU General Public License v3.0

%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import os
import pickle
import scipy
import datetime
import numpy as np
import tensorflow as tf

seed = 1
tf.random.set_seed(seed)
np.random.seed(seed)

In [ ]:
# Libraries for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.manifold

sns.set_context('poster')

In [ ]:
# Library scripts
import raw_data_processing
import data_pre_processing
import simclr_models
import simclr_utitlities
import transformations

In [ ]:
working_directory = 'test_run/'
dataset_save_path = working_directory
if not os.path.exists(working_directory):
    os.mkdir(working_directory)

## WISDM Dataset

In this section, the WISDM dataset will be parsed.


In [ ]:
# Load WISDM
wisdm_file_path = 'datasets/WISDM/WISDM_ar_v1.1_raw.txt'
user_datasets = raw_data_processing.process_wisdm_raw_file(wisdm_file_path)

## Pre-processing

In [ ]:
# Parameters
# WISDM: 20Hz sampling rate, 160 = 8 seconds window (same as MotionSense 50Hz 400)
window_size = 160
input_shape = (window_size, 3)

# Dataset Metadata 
transformation_multiple = 1
dataset_name = 'wisdm.pkl'
dataset_name_user_split = 'wisdm_user_split.pkl'

# WISDM labels (original lowercase)
label_list = ['null', 'Walking', 'Jogging', 'Sitting', 'Standing', 'Upstairs', 'Downstairs']
# Use this if labels are lowercase in raw data: ['null', 'walking', 'jogging', 'sitting', 'standing', 'upstairs', 'downstairs']
label_map = dict([(l, i) for i, l in enumerate(label_list)])
output_shape = len(label_list)

has_null_class = True
model_save_name = "wisdm_acc"

# Fixed user split: take every 5th user as test
def get_fixed_split_users(har_users):
    test_users = har_users[0::5]
    train_users = [u for u in har_users if u not in test_users]
    return (train_users, test_users)

In [ ]:
with open(dataset_save_path + dataset_name_user_split, 'wb') as f:
    pickle.dump({
        'user_split': user_datasets,
    }, f)

In [ ]:
har_users = list(user_datasets.keys())
train_users, test_users = get_fixed_split_users(har_users)
print(f'Testing: {test_users}, Training: {train_users}')

In [ ]:
np_train, np_val, np_test = data_pre_processing.pre_process_dataset_composite(
    user_datasets=user_datasets, 
    label_map=label_map, 
    output_shape=output_shape, 
    train_users=train_users, 
    test_users=test_users, 
    window_size=window_size, 
    shift=window_size//2, 
    normalise_dataset=True, 
    verbose=1
)


## SimCLR Training

In [ ]:
batch_size = 512
decay_steps = 1000
epochs = 200
temperature = 0.1
transform_funcs = [
    # transformations.scaling_transform_vectorized, # Use Scaling transformation
    transformations.rotation_transform_vectorized # Use rotation transformation
]
transformation_function = simclr_utitlities.generate_composite_transform_function_simple(transform_funcs)

In [ ]:
start_time = datetime.datetime.now()
start_time_str = start_time.strftime("%Y%m%d-%H%M%S")
tf.keras.backend.set_floatx('float32')

lr_decayed_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=0.1, decay_steps=decay_steps)
optimizer = tf.keras.optimizers.SGD(learning_rate=lr_decayed_fn)

base_model = simclr_models.create_base_model(input_shape, model_name="base_model")
simclr_model = simclr_models.attach_simclr_head(base_model)
simclr_model.summary()

trained_simclr_model, epoch_losses = simclr_utitlities.simclr_train_model(simclr_model, np_train[0], optimizer, batch_size, transformation_function, temperature=temperature, epochs=epochs, is_trasnform_function_vectorized=True, verbose=1)

simclr_model_save_path = f"{working_directory}{start_time_str}_simclr.keras"
trained_simclr_model.save(simclr_model_save_path)

In [ ]:
plt.figure(figsize=(12,8))
plt.plot(epoch_losses)
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.show()

## Fine-tuning and Evaluation

### Linear Model

In [ ]:
total_epochs = 50
batch_size = 200

tag = "linear_eval"

simclr_model = tf.keras.models.load_model(simclr_model_save_path)
linear_evaluation_model = simclr_models.create_linear_model_from_base_model(simclr_model, output_shape, intermediate_layer=7)

linear_eval_best_model_file_name = f"{working_directory}{start_time_str}_simclr_{tag}.keras"
best_model_callback = tf.keras.callbacks.ModelCheckpoint(linear_eval_best_model_file_name,
    monitor='val_loss', mode='min', save_best_only=True, save_weights_only=False, verbose=0
)

training_history = linear_evaluation_model.fit(
    x = np_train[0],
    y = np_train[1],
    batch_size=batch_size,
    shuffle=True,
    epochs=total_epochs,
    callbacks=[best_model_callback],
    validation_data=np_val
)

linear_eval_best_model = tf.keras.models.load_model(linear_eval_best_model_file_name)

print("Model with lowest validation Loss:")
print(simclr_utitlities.evaluate_model_simple(linear_eval_best_model.predict(np_test[0]), np_test[1], return_dict=True))
print("Model in last epoch")
print(simclr_utitlities.evaluate_model_simple(linear_evaluation_model.predict(np_test[0]), np_test[1], return_dict=True))

### Full HAR Model

In [ ]:
total_epochs = 50
batch_size = 200
tag = "full_eval"

simclr_model = tf.keras.models.load_model(simclr_model_save_path)
full_evaluation_model = simclr_models.create_full_classification_model_from_base_model(simclr_model, output_shape, model_name="TPN", intermediate_layer=7, last_freeze_layer=4)

full_eval_best_model_file_name = f"{working_directory}{start_time_str}_simclr_{tag}.keras"
best_model_callback = tf.keras.callbacks.ModelCheckpoint(full_eval_best_model_file_name,
    monitor='val_loss', mode='min', save_best_only=True, save_weights_only=False, verbose=0
)

training_history = full_evaluation_model.fit(
    x = np_train[0],
    y = np_train[1],
    batch_size=batch_size,
    shuffle=True,
    epochs=total_epochs,
    callbacks=[best_model_callback],
    validation_data=np_val
)

full_eval_best_model = tf.keras.models.load_model(full_eval_best_model_file_name)

print("Model with lowest validation Loss:")
print(simclr_utitlities.evaluate_model_simple(full_eval_best_model.predict(np_test[0]), np_test[1], return_dict=True))
print("Model in last epoch")
print(simclr_utitlities.evaluate_model_simple(full_evaluation_model.predict(np_test[0]), np_test[1], return_dict=True))

## Extra: t-SNE Plots

### Parameters

In [ ]:
# Select a model from which the intermediate representations are extracted
target_model = simclr_model 
perplexity = 30.0

### t-SNE Representations

In [ ]:
intermediate_model = simclr_models.extract_intermediate_model_from_base_model(target_model, intermediate_layer=7)
intermediate_model.summary()

embeddings = intermediate_model.predict(np_test[0], batch_size=600)
tsne_model = sklearn.manifold.TSNE(perplexity=perplexity, verbose=1, random_state=42)
tsne_projections = tsne_model.fit_transform(embeddings)

### Plotting

In [ ]:
categories = np.argmax(np_test[1], axis=1)
plt.figure(figsize=(16, 12))
sns.scatterplot(x=tsne_projections[:,0], y=tsne_projections[:,1], hue=categories, palette='tab10', s=50)
plt.legend(range(len(label_list)), label_list)
plt.show()